# GAT 포스터용 결과 (4번 표·5번 구간표·9번 SHAP)
1. GAT 단독 본 학습 다시 실행 (시드 0~4, heads 8 / lr 0.01 / dropout 0.3 / wd 5e-4 / patience 100 / 최대 500) → PR-AUC·F1·정밀도·재현율
2. 테스트 구간별 PR-AUC (35~42 / 43~49)
3. SHAP 판단 근거 비중 (원본만 / OOF 임베딩 / 누수 임베딩) — 팀 `analysis/shap_compare.py` 그대로 사용

**준비:** 팀 표준 노트북을 돌렸을 때 드라이브에 생긴 `gat_oof_results/emb_oof/` 폴더가 있어야 합니다 (SHAP용 OOF 임베딩).
**런타임 > 런타임 유형 변경 > T4 GPU** 후 위에서부터 실행하세요.

In [ ]:
# 1) 저장소(main) + GAT 기준 임베딩 + 패키지
!nvidia-smi -L || echo 'GPU 없음 -> 런타임 유형을 T4 GPU로 바꾸세요'
import getpass, os
REPO = 'github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git'
tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
host = f'{tok}@' if tok else ''
if not os.path.exists('/content/2026-2-HAI-ProjectTeam8'):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q -b main https://{host}{REPO} /content/2026-2-HAI-ProjectTeam8
%cd /content/2026-2-HAI-ProjectTeam8
auth = f'-H "Authorization: token {tok}"' if tok else ''
for s in range(5):   # GAT 기준 임베딩(LFS)을 직접 받기 → SHAP '누수 임베딩'용
    f = f'results/emb/emb_gat_seed{s}.npy'
    url = f'https://media.githubusercontent.com/media/HanyangTechAI/2026-2-HAI-ProjectTeam8/main/results/emb/emb_gat_seed{s}.npy'
    !curl -sfL {auth} -o {f} {url}
!ls -la results/emb/ | grep gat   # 각각 약 52MB 여야 함
!pip install -q torch-geometric "xgboost>=3.4.1,<3.5" shap

In [ ]:
# 2) Elliptic 데이터 자동 다운로드
from torch_geometric.datasets import EllipticBitcoinDataset
EllipticBitcoinDataset(root='data/elliptic')
!python scripts/verify_data.py data/elliptic

In [ ]:
%%writefile models/gat.py
"""GAT (Veličković et al. 2018).  계획서 4장 공통 사양: 2층 conv + 선형 분류기, 임베딩 64차원.

  python -m models.gat --data-root data/elliptic --seeds 0 1 2 3 4
  python -m models.gat --data-root data/elliptic --seeds 0 --heads 8 --results-dir results/tune/gat_h8   # 튜닝 예

models/gcn.py를 복사해서 conv 층만 GATConv로 바꾼 파일.
  - heads=8 기본(기존 GAT 최종 설정: heads 8, lr 0.01, dropout 0.3), 헤드당 8차원 → concat 64차원.
  - hybrid/make_oof_emb.py는 GAT(in_dim, hidden, dropout)만 넘기므로 이 기본값(8)이 그대로 쓰인다.
  - 헤드 8개로 바꿔도 헤드당 8차원 × 8 = 64차원이라 임베딩 크기는 그대로.
  - GATConv의 dropout 인자 = 어텐션 계수(이웃 연결)에 거는 드롭아웃.
  - --heads는 common/train_gnn.py에 없는 GAT 전용 인자라, 여기서 먼저 읽고 나머지 인자만 main()에 넘긴다
    (공통 코드는 수정하지 않음).
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.train_gnn import main  # noqa: E402


class GAT(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 64, dropout: float = 0.5, heads: int = 8):
        super().__init__()
        assert hidden % heads == 0, "hidden은 heads로 나누어떨어져야 함"
        per_head = hidden // heads
        self.conv1 = GATConv(in_dim, per_head, heads=heads, dropout=dropout)   # → per_head × heads = 64
        self.conv2 = GATConv(hidden, per_head, heads=heads, dropout=dropout)   # 이 층의 출력 = 임베딩 (64차원)
        self.head = nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = F.elu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        emb = self.conv2(h, edge_index)
        logits = self.head(F.dropout(F.elu(emb), p=self.dropout, training=self.training))
        return logits, emb


if __name__ == "__main__":
    pre = argparse.ArgumentParser(add_help=False)
    pre.add_argument("--heads", type=int, default=8, help="어텐션 헤드 수 (튜닝: 4 또는 8)")
    gat_args, rest = pre.parse_known_args()
    # GAT 기본값: patience 100, 최대 500에폭 (실행할 때 --patience / --max-epochs를 주면 그 값이 우선)
    if not any(a.startswith("--patience") for a in rest):
        rest += ["--patience", "100"]
    if not any(a.startswith("--max-epochs") for a in rest):
        rest += ["--max-epochs", "500"]
    print(f"GAT heads={gat_args.heads}, args={rest}")
    main("gat", lambda in_dim, a: GAT(in_dim, a.hidden, a.dropout, gat_args.heads), argv=rest)


In [ ]:
# 3) GAT 단독 본 학습 (시드 0~4, 가장 오래 걸림). 결과는 results_gat/ 에 (저장소 기존 파일은 안 건드림)
!python -m models.gat --data-root data/elliptic --seeds 0 1 2 3 4 --heads 8 --lr 0.01 --dropout 0.3 --weight-decay 5e-4 --patience 100 --max-epochs 500 --results-dir results_gat
# 확인: best_epoch가 100, 330, 159, 405, 122 와 같으면 저장소 기준 임베딩과 같은 실행이 재현된 것
import pandas as pd, re
g = pd.read_csv('results_gat/metrics/gat.csv', dtype={'threshold_type': str})
print('best_epoch:', [int(re.search(r'best_epoch=(\d+)', n).group(1)) for n in g[(g.split=='test')&(g.threshold_type=='fixed_0.5')].sort_values('seed').notes])

In [ ]:
# 4) 포스터 4번 표: GAT 단독 (테스트, 임계값 0.5 — 포스터 표와 같은 기준)
t = g[(g.split=='test')&(g.threshold_type=='fixed_0.5')]
print(f"GAT  PR-AUC {t.pr_auc.mean():.3f} ± {t.pr_auc.std():.3f} | F1 {t.f1.mean():.3f} | 정밀도 {t.precision.mean():.3f} | 재현율 {t.recall.mean():.3f}")
print('시드별 PR-AUC:', t.sort_values('seed').pr_auc.round(4).tolist())

# 5) 포스터 5번 표: 테스트 구간별 PR-AUC (시드별 계산 후 평균). XGBoost·GCN도 같은 방식으로 다시 계산해 포스터 값과 대조
import numpy as np
from sklearn.metrics import average_precision_score as AP
from common.data import load_data
d = load_data('data/elliptic', standardize=False)
te, ts, y = d.test_mask, d.time_step, d.y
segs = {'35~42': te & (ts <= 42), '43~49': te & (ts >= 43)}
def seg(prefix):
    out = {}
    for k, m in segs.items():
        v = [AP(y[m], np.load(f'{prefix}{s}.npy')[m]) for s in range(5)]
        out[k] = f'{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}'
    return out
!git lfs pull --include="results/pred/*" 2>/dev/null || true
for name, prefix in [('XGBoost', 'results/pred/pred_xgb_seed'), ('GCN', 'results/pred/pred_gcn_seed'), ('GAT', 'results_gat/pred/pred_gat_seed')]:
    try: print(name, seg(prefix))
    except Exception as e: print(name, '계산 실패:', e)

In [ ]:
# 6) 포스터 9번 SHAP: 드라이브의 GAT OOF 임베딩 사용 (원본만 / OOF 임베딩 / 누수 임베딩, OOF 확률은 GAT 방식 B에 없어서 제외)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p results/oof
for s in range(5):
    !cp "/content/drive/MyDrive/gat_oof_results/emb_oof/emb_oof_gat_seed{s}.npy" results/oof/oofemb_gat_seed{s}.npy
!ls -la results/oof/
# 팀 코드에서 'OOF 확률' 한 줄만 뺀 복사본으로 실행 (원본 파일은 그대로)
!sed '/"hybrid_oofprob": lambda/d' analysis/shap_compare.py > analysis/shap_compare_gat.py
!python -m analysis.shap_compare_gat --gnn gat --oof-dir results/oof

In [ ]:
# 7) 결과 요약 -> 이 셀 출력을 Claude에게 붙여 주세요
s = pd.read_csv('results/shap/gat/group_share.csv')
print((s.groupby('model')[['local', 'agg', 'gnn']].mean() * 100).round(1).to_string())
print(t[['seed', 'pr_auc', 'f1', 'precision', 'recall']].round(4).to_string(index=False))

In [ ]:
# 8) 작은 결과 zip 내려받기 (Claude에게 올려 주세요) + 드라이브 백업
!mkdir -p /content/gat_poster && cp -r results_gat/metrics results_gat/pred results_gat/timestep_f1_gat.csv results/shap/gat /content/gat_poster/ 2>/dev/null
!cd /content && zip -qr gat_poster.zip gat_poster && cp -r /content/gat_poster "/content/drive/MyDrive/" && ls -la /content/gat_poster.zip
from google.colab import files; files.download('/content/gat_poster.zip')